# Pixabay — Descrição de Imagens via IA

Preenche na planilha **pixabay-image-stock** (aba `image-stock`), usando Groq e Mistral (visão) com rodízio entre os dois provedores:

- `Tags_PT` / `Tags_EN` — tags originais do Pixabay, limpas/traduzidas
- `Descricao_Cena_PT` / `Descricao_Cena_EN` — 2 frases do que se vê na imagem
- `Tags_Descricao_PT` / `Tags_Descricao_EN` — palavras-chave extraídas da descrição que não estavam em Tags_PT (mesma chamada de IA, sem custo extra)
- `Tags_Semelhantes_PT` / `Tags_Semelhantes_EN` — Tags_PT + Tags_Descricao, expandidas por sinônimo (dicionário embutido, sem IA) -- é isso que o painel de revisão usa pra achar candidato pra cada versículo

**Essa planilha agora é dedicada só ao projeto de narração bíblica** -- não pede mais tags de oração/devocional nem tags de tema bíblico fechado (isso saiu do layout; oração vai ganhar sua própria planilha no futuro, e o tema bíblico agora é resolvido por match de tags, não lista fechada).

Roda direto no Google Sheets — não precisa baixar nada manualmente, só rodar as células em ordem.

**URL fresca**: a URL de imagem guardada na planilha (coluna `Imagem`) só vale 24h — esse notebook busca uma URL nova direto na API da Pixabay (usando o ID, que é permanente) antes de cada download.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. DEPENDÊNCIAS E AUTENTICAÇÃO                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread requests "mistralai>=1.2.0"

import json
import time
from pathlib import Path

import gspread
import requests
from google.auth import default
from google.colab import auth, userdata
from groq import Groq
# Caminho de import oficial e documentado do SDK unificado da Mistral (não é
# "mistralai import Mistral" — veja o próprio exemplo de vision da Mistral em
# https://docs.mistral.ai/capabilities/vision):
from mistralai.client import Mistral

# Autenticação Google Drive / Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Chaves via Secrets do Colab (ícone de chave 🔑 na barra lateral esquerda)
GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
PIXABAY_API_KEY = userdata.get("PIXABAY_KEY")  # gratuita, cadastro em pixabay.com/api/docs

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

print("=" * 60)
print("✅ SETUP CONCLUÍDO")
print("=" * 60)
print(f"   Groq:    {'disponível' if groq_client else '❌ GROQ_KEY não encontrada nos Secrets'}")
print(f"   Mistral: {'disponível' if mistral_client else '❌ MISTRAL_KEY não encontrada nos Secrets'}")
print(f"   Pixabay: {'disponível' if PIXABAY_API_KEY else '❌ PIXABAY_KEY não encontrada nos Secrets (necessária pra buscar URL fresca de imagem)'}")
print("=" * 60)

✅ SETUP CONCLUÍDO
   Groq:    disponível
   Mistral: disponível
   Pixabay: disponível


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1b. DICIONÁRIO DE SINÔNIMOS (embutido -- notebook standalone)   ║
# ╚══════════════════════════════════════════════════════════════════╝
# Esse notebook não monta o Drive nem copia pipeline/modulos/ (é
# standalone de propósito) -- por isso o dicionário vem colado aqui
# em vez de importado. É o MESMO conteúdo de
# pipeline/modulos/dicionario_sinonimos.py -- se editar um, edita o
# outro também, pra não desincronizar.

# -*- coding: utf-8 -*-
"""
dicionario_sinonimos.py — Grupos de palavras equivalentes, PT e EN, pra
expandir Tags_Semelhantes_PT/EN tanto do lado da Bíblia (versículo/
título/evento) quanto do lado das imagens (Tags_PT/EN da image-stock).

Cada grupo é uma lista de palavras/expressões que devem "casar" entre si
no match -- ex: uma imagem taggeada "bebê" deve aparecer como candidata
pra um versículo taggeado "criança", mesmo sem a palavra exata bater.
Os grupos PT e EN estão na MESMA ORDEM (grupo N do PT corresponde ao
grupo N do EN) -- não é usado pra tradução aqui, só mantém os dois
dicionários organizados/fáceis de conferir lado a lado.

Curado manualmente, focado em vocabulário visual/bíblico -- vai crescer
conforme aparecerem mais casos reais na revisão.
"""

GRUPOS_SINONIMOS_PT = [
    # ── pessoas / família ──────────────────────────────────────────────
    ["criança", "crianças", "bebê", "bebês", "infante", "menino", "menina", "filho", "filha", "filhos"],
    ["jovem", "jovens", "rapaz", "moça", "adolescente"],
    ["idoso", "idosos", "velho", "velha", "ancião", "anciã", "anciãos"],
    ["homem", "homens", "varão"],
    ["mulher", "mulheres"],
    ["família", "famílias", "parentes", "parentesco"],
    ["irmão", "irmãos", "irmã", "irmãs"],
    ["multidão", "multidões", "povo", "gente", "massa", "aglomeração"],
    ["pai", "pais", "progenitor"],
    ["mãe", "mães", "progenitora"],

    # ── autoridade / religião ────────────────────────────────────────
    ["rei", "reis", "monarca", "soberano", "governante"],
    ["rainha", "rainhas"],
    ["sacerdote", "sacerdotes", "padre", "clérigo", "sumo sacerdote"],
    ["profeta", "profetas", "profetisa"],
    ["anjo", "anjos", "mensageiro celestial", "ser celestial"],
    ["discípulo", "discípulos", "apóstolo", "apóstolos", "seguidor", "seguidores"],
    ["pastor", "pastores", "cuidador de ovelhas"],
    ["escriba", "escribas", "fariseu", "fariseus", "doutor da lei"],
    ["soldado", "soldados", "guerreiro", "guerreiros", "guarda", "guardas"],

    # ── natureza / lugar ────────────────────────────────────────────
    ["deserto", "deserto árido", "ermo", "terra árida", "areia"],
    ["montanha", "montanhas", "monte", "montes", "colina", "colinas"],
    ["mar", "oceano", "águas profundas"],
    ["rio", "rios", "riacho", "córrego", "ribeirão"],
    ["céu", "céus", "firmamento"],
    ["estrela", "estrelas", "astro", "constelação"],
    ["noite", "escuridão", "trevas", "anoitecer"],
    ["dia", "luz do dia", "amanhecer", "alvorecer", "aurora"],
    ["luz", "claridade", "brilho", "resplendor", "fulgor"],
    ["nuvem", "nuvens", "nevoeiro"],
    ["chuva", "tempestade", "temporal"],
    ["árvore", "árvores", "floresta", "bosque"],
    ["campo", "campos", "plantação", "lavoura", "seara"],
    ["caverna", "gruta", "cova", "gruta rochosa"],

    # ── estruturas / lugares construídos ──────────────────────────
    ["templo", "santuário", "casa de oração", "lugar sagrado"],
    ["casa", "lar", "moradia", "residência", "habitação"],
    ["cidade", "cidades", "vila", "povoado", "vilarejo"],
    ["palácio", "palácios", "corte real"],
    ["prisão", "cárcere", "masmorra", "cadeia"],
    ["tumba", "túmulo", "sepulcro", "sepultura"],
    ["cruz", "crucifixo"],
    ["altar", "altares"],
    ["trono", "tronos"],

    # ── ações ──────────────────────────────────────────────────────
    ["caminhar", "andar", "viajar", "jornada", "peregrinação", "travessia"],
    ["orar", "rezar", "suplicar", "clamar", "invocar"],
    ["curar", "cura", "curando", "curado", "saúde restaurada"],
    ["ensinar", "ensinando", "pregar", "pregando", "prédica", "sermão"],
    ["chorar", "choro", "lágrimas", "lamento", "pranto"],
    ["celebrar", "celebração", "festa", "banquete", "festejo"],
    ["adorar", "adoração", "louvor", "louvar", "reverência"],
    ["fugir", "fuga", "fugindo", "escapar", "refúgio"],

    # ── objetos ────────────────────────────────────────────────────
    ["espada", "espadas", "arma", "armas"],
    ["coroa", "coroas", "diadema"],
    ["pão", "pães", "alimento", "comida"],
    ["vinho", "taça", "cálice"],
    ["barco", "barcos", "navio", "embarcação"],
    ["lâmpada", "lâmpadas", "candeeiro", "vela", "chama"],
    ["manjedoura", "presépio", "berço"],

    # ── animais ────────────────────────────────────────────────────
    ["ovelha", "ovelhas", "cordeiro", "cordeiros", "rebanho"],
    ["leão", "leões"],
    ["burro", "jumento", "asno"],
    ["peixe", "peixes"],
    ["pomba", "pombas", "ave", "aves", "pássaro", "pássaros"],

    # ── emoções / estados ──────────────────────────────────────────
    ["alegria", "felicidade", "contentamento", "júbilo", "regozijo"],
    ["tristeza", "dor", "sofrimento", "angústia", "aflição"],
    ["medo", "temor", "pavor", "terror"],
    ["paz", "tranquilidade", "serenidade", "calma"],
    ["esperança", "confiança", "fé"],
    ["milagre", "milagroso", "prodígio", "maravilha"],
]

GRUPOS_SINONIMOS_EN = [
    # ── people / family ──────────────────────────────────────────────
    ["child", "children", "baby", "babies", "infant", "boy", "girl", "son", "daughter", "kid", "kids"],
    ["youth", "young man", "young woman", "teenager", "adolescent"],
    ["elder", "elders", "old man", "old woman", "elderly"],
    ["man", "men"],
    ["woman", "women"],
    ["family", "families", "relatives", "kin"],
    ["brother", "brothers", "sister", "sisters"],
    ["crowd", "crowds", "people", "multitude", "throng"],
    ["father", "fathers", "dad"],
    ["mother", "mothers", "mom"],

    # ── authority / religion ────────────────────────────────────────
    ["king", "kings", "monarch", "sovereign", "ruler"],
    ["queen", "queens"],
    ["priest", "priests", "clergyman", "high priest"],
    ["prophet", "prophets", "prophetess"],
    ["angel", "angels", "heavenly messenger", "celestial being"],
    ["disciple", "disciples", "apostle", "apostles", "follower", "followers"],
    ["shepherd", "shepherds"],
    ["scribe", "scribes", "pharisee", "pharisees", "teacher of the law"],
    ["soldier", "soldiers", "warrior", "warriors", "guard", "guards"],

    # ── nature / place ────────────────────────────────────────────
    ["desert", "arid desert", "wilderness", "dry land", "sand"],
    ["mountain", "mountains", "mount", "hill", "hills"],
    ["sea", "ocean", "deep waters"],
    ["river", "rivers", "stream", "creek", "brook"],
    ["sky", "heaven", "heavens", "firmament"],
    ["star", "stars", "constellation"],
    ["night", "darkness", "dusk", "nightfall"],
    ["day", "daylight", "dawn", "sunrise"],
    ["light", "brightness", "radiance", "glow"],
    ["cloud", "clouds", "fog", "mist"],
    ["rain", "storm", "tempest"],
    ["tree", "trees", "forest", "woods"],
    ["field", "fields", "crop", "farmland", "harvest"],
    ["cave", "grotto", "cavern", "rocky cave"],

    # ── structures / built places ──────────────────────────────
    ["temple", "sanctuary", "house of prayer", "holy place"],
    ["house", "home", "dwelling", "residence", "abode"],
    ["city", "cities", "village", "town"],
    ["palace", "palaces", "royal court"],
    ["prison", "jail", "dungeon"],
    ["tomb", "grave", "sepulcher", "sepulchre"],
    ["cross", "crucifix"],
    ["altar", "altars"],
    ["throne", "thrones"],

    # ── actions ──────────────────────────────────────────────────────
    ["walk", "walking", "travel", "journey", "pilgrimage", "crossing"],
    ["pray", "praying", "plead", "beg", "invoke"],
    ["heal", "healing", "healed", "restored health"],
    ["teach", "teaching", "preach", "preaching", "sermon"],
    ["cry", "crying", "tears", "weeping", "mourning"],
    ["celebrate", "celebration", "feast", "banquet", "festivity"],
    ["worship", "adoration", "praise", "reverence"],
    ["flee", "flight", "fleeing", "escape", "refuge"],

    # ── objects ────────────────────────────────────────────────────
    ["sword", "swords", "weapon", "weapons"],
    ["crown", "crowns", "diadem"],
    ["bread", "loaves", "food"],
    ["wine", "cup", "chalice", "goblet"],
    ["boat", "boats", "ship", "vessel"],
    ["lamp", "lamps", "candle", "candlestick", "flame"],
    ["manger", "nativity crib", "cradle"],

    # ── animals ────────────────────────────────────────────────────
    ["sheep", "lamb", "lambs", "flock"],
    ["lion", "lions"],
    ["donkey", "colt", "ass"],
    ["fish", "fishes"],
    ["dove", "doves", "bird", "birds"],

    # ── emotions / states ──────────────────────────────────────────
    ["joy", "happiness", "gladness", "delight", "rejoicing"],
    ["sadness", "sorrow", "suffering", "anguish", "distress"],
    ["fear", "dread", "terror"],
    ["peace", "tranquility", "serenity", "calm"],
    ["hope", "trust", "faith"],
    ["miracle", "miraculous", "wonder", "marvel"],
]


def _construir_mapa(grupos):
    import unicodedata

    def normalizar(s):
        nfkd = unicodedata.normalize("NFD", s.lower().strip())
        return "".join(c for c in nfkd if not unicodedata.combining(c))

    mapa = {}
    for grupo in grupos:
        for palavra in grupo:
            mapa[normalizar(palavra)] = grupo  # guarda o grupo ORIGINAL (com acento, pra exibir bonito)
    return mapa


MAPA_SINONIMOS_PT = _construir_mapa(GRUPOS_SINONIMOS_PT)
MAPA_SINONIMOS_EN = _construir_mapa(GRUPOS_SINONIMOS_EN)


def expandir_tags_semelhantes(tags, idioma="pt"):
    """
    Recebe uma lista (ou string separada por vírgula) de tags e devolve
    a lista expandida -- cada tag que tem sinônimo conhecido traz o
    GRUPO INTEIRO junto (a tag original sempre continua incluída).

    idioma: "pt" ou "en" -- escolhe qual dicionário usar.

    Determinístico, sem IA -- só dicionário. Pode rodar pra Bíblia
    inteira (título/evento) de uma vez, sem custo de API nem espera.
    """
    import unicodedata

    def normalizar(s):
        nfkd = unicodedata.normalize("NFD", s.lower().strip())
        return "".join(c for c in nfkd if not unicodedata.combining(c))

    mapa = MAPA_SINONIMOS_EN if idioma == "en" else MAPA_SINONIMOS_PT

    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]

    expandido = list(dict.fromkeys(tags))  # preserva original, remove duplicata mantendo ordem
    for tag in tags:
        chaves_a_testar = [normalizar(tag)]
        # tag pode ser uma FRASE ("rei Herodes", "cidade de Jerusalém") -- o
        # dicionário é de PALAVRA SOLTA ("rei", "cidade"), então só testar a
        # frase inteira quase nunca bate. Testa cada palavra da frase também
        # (sem duplicar o teste se a "frase" já for uma palavra só).
        palavras = normalizar(tag).split()
        if len(palavras) > 1:
            chaves_a_testar += palavras
        for chave in chaves_a_testar:
            grupo = mapa.get(chave)
            if grupo:
                for sinonimo in grupo:
                    if sinonimo not in expandido:
                        expandido.append(sinonimo)
    return expandido


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
ID_PLANILHA = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"  # ID fixo da planilha (parte
                            # da URL entre /d/ e /edit) — abre por ID em vez de nome, porque a
                            # planilha está numa conta Google diferente da que autentica esse
                            # notebook (alanabdmorais@gmail.com vs oracaobomdiajesus@gmail.com).
NOME_ABA            = "image-stock"  # aba dentro da planilha

MODELO_GROQ    = "qwen/qwen3.6-27b"          # confirmado em console.groq.com/docs/vision (ago/2026)
MODELO_MISTRAL = "mistral-small-latest"      # alias oficial (docs.mistral.ai/capabilities/vision) — sempre aponta pro modelo de vision atual

DELAY_SEGUNDOS      = 6    # pausa entre linhas -- protege tanto a IA quanto as 2 chamadas Pixabay por linha (URL fresca + download)
TIMEOUT_DOWNLOAD    = 15   # segundos, download da imagem
MAX_TOKENS_RESPOSTA = 1200  # tokens de saída — margem confortável pros 8 campos

# Quantas linhas AINDA PENDENTES (sem Descricao_Cena_PT) processar nesta
# execução — não conta as que já estão preenchidas. Comece baixo pra
# testar; aumente (ou deixe None pra processar todas as pendentes) depois
# que confirmar que está tudo certo.
LIMITE_LINHAS = 100

print(f"Planilha (ID): {ID_PLANILHA} / aba: {NOME_ABA}")
print(f"Delay entre linhas: {DELAY_SEGUNDOS}s")
print(f"Limite desta execução: {LIMITE_LINHAS if LIMITE_LINHAS is not None else 'sem limite — todas as pendentes'}")

Planilha (ID): 1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E / aba: image-stock
Delay entre linhas: 6s
Limite desta execução: 100


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. ABRIR A PLANILHA E DIAGNÓSTICO                               ║
# ╚══════════════════════════════════════════════════════════════════╝
sheet = gc.open_by_key(ID_PLANILHA).worksheet(NOME_ABA)
dados_tabela = sheet.get_all_records()
cabecalho = sheet.row_values(1)


def col(nome):
    """Índice 1-based da coluna pelo nome do cabeçalho — se a ordem das
    colunas mudar na planilha um dia, isso continua funcionando (não
    depende de posição fixa)."""
    return cabecalho.index(nome) + 1


# Colunas que este notebook escreve -- agora que Tags_Biblia_*/Tags_Oracao_*
# saíram da planilha (essa image-stock ficou dedicada só à narração
# bíblica), não tem mais "buraco" no meio pra pular. Ainda assim, escreve
# via batch_update (uma chamada só, célula por célula) em vez de assumir
# que essas colunas são contíguas -- fica robusto a qualquer ordem de
# coluna que a planilha tenha.
COLUNAS_ESTE_NOTEBOOK = [
    "Tags_PT", "Tags_EN", "Descricao_Cena_PT", "Descricao_Cena_EN",
    "Tags_Descricao_PT", "Tags_Descricao_EN",
    "Tags_Semelhantes_PT", "Tags_Semelhantes_EN",
]

# cria as colunas que ainda não existem (expande a grade da planilha antes
# se precisar -- senão dá erro "exceeds grid limits", já vimos esse erro
# antes no pixabay-image-seed.ipynb)
faltando = [c for c in COLUNAS_ESTE_NOTEBOOK if c not in cabecalho]
if faltando:
    if len(cabecalho) + len(faltando) > sheet.col_count:
        sheet.add_cols(len(cabecalho) + len(faltando) - sheet.col_count)
    for i, nome in enumerate(faltando):
        sheet.update_cell(1, len(cabecalho) + 1 + i, nome)
    cabecalho = cabecalho + faltando

ja_preenchidas = sum(1 for r in dados_tabela if r.get("Descricao_Cena_PT"))
pendentes = len(dados_tabela) - ja_preenchidas

print("=" * 60)
print("📊 DIAGNÓSTICO")
print("=" * 60)
print(f"   Total de linhas: {len(dados_tabela)}")
print(f"   Já preenchidas:  {ja_preenchidas}")
print(f"   Pendentes:       {pendentes}")
print("=" * 60)


📊 DIAGNÓSTICO
   Total de linhas: 946
   Já preenchidas:  863
   Pendentes:       83


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. FUNÇÕES                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
PROMPT_SISTEMA = """
Analise a imagem e retorne JSON estrito com os campos abaixo.
Respeite os limites à risca — respostas longas demais quebram o pipeline.

- tags_pt: tradução/ajuste das tags originais para PT-BR. NO MÁXIMO as mesmas
  tags originais, sem adicionar novas. Separadas por vírgula, sem frases.
- tags_en: tags originais limpas, sem termos genéricos. MESMO LIMITE de tags_pt.

- descricao_cena_pt: EXATAMENTE 2 frases curtas e literais (o que se vê, sem
  poesia). Máximo 40 palavras no total.
- descricao_cena_en: tradução da descrição. MESMO LIMITE de 40 palavras.

- tags_descricao_pt: 2 a 5 palavras/expressões CURTAS que capturem elementos
  visuais mencionados na descricao_cena_pt que NÃO estejam já em tags_pt
  (ex: se a descrição menciona "pôr do sol" e isso não é uma tag original,
  inclua aqui). Se não houver nada novo, deixe vazio -- não repita tags_pt.
- tags_descricao_en: tradução direta de cada item de tags_descricao_pt.

Responda SÓ o JSON, sem texto antes ou depois. Este notebook é só pra
descrição visual da imagem -- não pede mais tags de oração/devocional
(essa planilha agora é dedicada ao projeto de narração bíblica; oração
vai ganhar sua própria planilha/tags no futuro).
"""


def _requisitar_com_retry(url, params=None, tentativas=4, timeout=TIMEOUT_DOWNLOAD):
    """GET com retry/backoff pra 429 (Too Many Requests) -- Pixabay está
    devolvendo 429 tanto na API quanto no download da imagem em si (CDN
    "pixabay.com/get/...") quando as requisições vêm rápido demais, mesmo
    com DELAY_SEGUNDOS entre LINHAS (isso não protege as 2 chamadas
    Pixabay DENTRO da mesma linha -- buscar_url_fresca + o download).
    Em vez de desistir na hora, espera mais a cada tentativa (5s, 10s,
    20s, 40s) e tenta de novo."""
    resp = None
    for tentativa in range(1, tentativas + 1):
        resp = requests.get(url, params=params, timeout=timeout)
        if resp.status_code != 429:
            return resp
        espera = 5 * (2 ** (tentativa - 1))
        print(f"  ⏳ 429 (Too Many Requests) da Pixabay -- esperando {espera}s (tentativa {tentativa}/{tentativas})")
        time.sleep(espera)
    return resp  # ainda 429 depois de todas as tentativas -- raise_for_status estoura la fora


def buscar_url_fresca(vid_id):
    """As URLs de imagem que a API da Pixabay devolve (webformatURL/
    largeImageURL) só valem por 24h -- por isso a URL guardada na planilha
    (coluna "Imagem") já pode estar vencida quando esse notebook roda. O ID
    da imagem, esse sim, é permanente -- usamos ele pra pedir uma URL nova
    direto na API antes de cada download.
    Doc oficial: https://pixabay.com/api/docs/"""
    if not PIXABAY_API_KEY:
        raise RuntimeError("PIXABAY_KEY não configurada nos Secrets do Colab")
    resp = _requisitar_com_retry(
        "https://pixabay.com/api/",
        params={"key": PIXABAY_API_KEY, "id": vid_id},
    )
    resp.raise_for_status()
    hits = resp.json().get("hits", [])
    if not hits:
        return None
    # webformatURL (640px) é suficiente pra descrição via IA -- mais leve
    # que largeImageURL, sem precisar da imagem em resolução alta
    return hits[0].get("webformatURL") or hits[0].get("largeImageURL")


def baixar_imagem(vid_id):
    url_fresca = buscar_url_fresca(vid_id)
    if not url_fresca:
        print(f"  ❌ ID {vid_id} não encontrado na Pixabay (removido ou inválido)")
        return None
    destino = Path(f"{vid_id}_foto.jpg")
    try:
        resp = _requisitar_com_retry(url_fresca)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ❌ Falha ao baixar imagem (URL fresca): {e}")
        return None
    destino.write_bytes(resp.content)
    return destino


def imagem_para_b64(caminho):
    import base64
    return base64.b64encode(caminho.read_bytes()).decode("utf-8")


def deletar_temp(*caminhos):
    for c in caminhos:
        if c and c.exists():
            try:
                c.unlink()
            except Exception:
                pass


def _extrair_json(texto):
    """Parse tolerante: tira cerca de código markdown (```json ... ```)
    se o modelo colocar uma, e só então tenta json.loads. Usado tanto pro
    Groq quanto pro Mistral -- ver comentário em chamar_groq sobre por que
    isso substituiu o response_format=json_object do Groq."""
    texto = texto.strip()
    if not texto:
        # resposta vazia (não é JSON malformado, é ausência de conteúdo --
        # visto acontecendo 100% das vezes no Groq numa rodada de teste;
        # mensagem própria pra não confundir com "JSON mal formatado")
        raise ValueError("Resposta vazia da IA (content='') -- não é erro de formato, é ausência de conteúdo")
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
        texto = texto.strip()
    return json.loads(texto)


def chamar_groq(prompt, b64_list):
    # Sem response_format=json_object de propósito: a validação ESTRITA do
    # Groq pro modo JSON está com falha de confiabilidade conhecida --
    # aqui a gente só pede JSON no texto do prompt e faz o parse tolerante
    # do nosso lado (_extrair_json).
    content = [{"type": "text", "text": prompt}]
    for img in b64_list:
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}})
    res = groq_client.chat.completions.create(
        model=MODELO_GROQ,
        messages=[{"role": "user", "content": content}],
        temperature=0.2,
        max_tokens=MAX_TOKENS_RESPOSTA,
    )
    return _extrair_json(res.choices[0].message.content)


def chamar_mistral(prompt, b64_list):
    content = [{"type": "text", "text": prompt}]
    for img in b64_list:
        content.append({"type": "image_url", "image_url": f"data:image/jpeg;base64,{img}"})
    res = mistral_client.chat.complete(
        model=MODELO_MISTRAL,
        messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object"},
        temperature=0.2,
        max_tokens=MAX_TOKENS_RESPOSTA,
    )
    return _extrair_json(res.choices[0].message.content)


def _e_cota_diaria_esgotada(erro):
    """Cota DIÁRIA (TPD — tokens per day) é diferente de cota por MINUTO
    (TPM): a diária só libera de novo depois de horas, não minutos — não
    vale a pena ficar tentando esse provedor de novo a cada linha pelo
    resto da execução."""
    texto = str(erro).lower()
    return "per day" in texto or "(tpd)" in texto or "tpd)" in texto


def _vale_tentar_outro_provedor(erro):
    """Heurística por texto no erro (+ checagem de tipo pro nosso próprio
    parse) — evita depender da classe exata de exceção de cada SDK.

    Cobre os tipos de falha que vale a pena tentar no OUTRO provedor:
    cota estourada (429/rate limit/quota), json_validate_failed (formato
    antigo do Groq, mantido por segurança), e ValueError -- cobre tanto
    json.JSONDecodeError (JSON malformado; é subclasse de ValueError)
    quanto "resposta vazia" (content="", ver _extrair_json -- visto
    acontecendo 100% das vezes no Groq numa rodada de teste; sem essa
    checagem, isso NÃO cairia pro Mistral, porque não bate nenhuma
    palavra-chave de cota)."""
    if isinstance(erro, ValueError):
        return True
    texto = str(erro).lower()
    e_erro_de_cota = any(m in texto for m in ("429", "rate limit", "rate_limit", "too many requests", "quota"))
    e_geracao_vazia = "json_validate_failed" in texto or "failed to validate json" in texto
    return e_erro_de_cota or e_geracao_vazia


LIMITE_FALHAS_VAZIAS_CONSECUTIVAS = 2  # depois disso, provedor vira indisponivel_ate_o_fim
                                        # (não é 1 pra tolerar 1 blip isolado, mas não deixa
                                        # ficar testando um provedor sistemicamente quebrado
                                        # em toda linha até o fim da execução)


def preencher_com_ia(prompt, b64_list, estado):
    """Tenta o provedor 'da vez'; se falhar por limite de cota, tenta
    IMEDIATAMENTE o outro antes de desistir da linha. Se um provedor bater
    na cota DIÁRIA -- ou devolver "resposta vazia" duas vezes seguidas
    (indício de falha sistêmica, ex: modelo sem suporte a visão de
    verdade, não é cota) -- marca ele como indisponível pro resto da
    execução, em vez de continuar gastando 1 chamada à toa em toda linha."""
    primeiro = estado["atual"]
    segundo = "mistral" if primeiro == "groq" else "groq"
    ultimo_erro = None

    for provedor in (primeiro, segundo):
        if estado.get("indisponivel_ate_o_fim", {}).get(provedor):
            continue
        cliente = groq_client if provedor == "groq" else mistral_client
        if not cliente:
            continue
        try:
            dados = chamar_groq(prompt, b64_list) if provedor == "groq" else chamar_mistral(prompt, b64_list)
            estado["atual"] = segundo if provedor == primeiro else primeiro
            estado.setdefault("falhas_vazias_consecutivas", {})[provedor] = 0  # zera a sequencia no sucesso
            return dados, provedor
        except Exception as e:
            ultimo_erro = e
            print(f"  ⚠️  Falha no {provedor}: {e}")
            if _e_cota_diaria_esgotada(e):
                estado.setdefault("indisponivel_ate_o_fim", {})[provedor] = True
                print(f"  🚫 {provedor}: cota DIÁRIA esgotada — não tenta mais ele nesta execução.")
            elif isinstance(e, ValueError) and "vazia" in str(e).lower():
                contador = estado.setdefault("falhas_vazias_consecutivas", {})
                contador[provedor] = contador.get(provedor, 0) + 1
                if contador[provedor] >= LIMITE_FALHAS_VAZIAS_CONSECUTIVAS:
                    estado.setdefault("indisponivel_ate_o_fim", {})[provedor] = True
                    print(f"  🚫 {provedor}: {contador[provedor]} respostas vazias seguidas — parece falha "
                          f"sistêmica (modelo sem suporte a visão? confira o nome do modelo), não tenta mais "
                          f"ele nesta execução.")
            if not _vale_tentar_outro_provedor(e):
                break

    estado["atual"] = segundo
    raise ultimo_erro or RuntimeError("Nenhum provedor de IA disponível — confira as chaves nos Secrets ou aguarde a cota liberar")


def normalizar_valor_celula(valor):
    """A IA às vezes devolve um campo como lista em vez de texto único —
    o Google Sheets rejeita lista dentro de uma célula. Junta em string
    nesse caso; qualquer outro tipo inesperado também vira string."""
    if isinstance(valor, (list, tuple)):
        return ", ".join(str(v) for v in valor)
    if valor is None:
        return ""
    return str(valor)


def gravar_linha(num_linha, valores_por_coluna, tentativas=3):
    """Grava várias colunas de UMA linha numa chamada só (batch_update),
    célula por célula -- não assume que as colunas são contíguas, então
    funciona não importa a ordem/posição delas na planilha.

    valores_por_coluna: dict {"Nome_Da_Coluna": valor} -- só as colunas
    passadas aqui são tocadas; qualquer coluna que não esteja no dict
    (ex: Tags_Biblia_PT se um dia voltar a existir) nunca é escrita nem
    apagada.

    IMPORTANTE: `corpo` é reconstruído em CADA tentativa, não reaproveitado
    entre retries -- o gspread MODIFICA o range dentro do próprio dict pra
    incluir o nome da aba (ex: "O154" vira "'image-stock'!O154"). Reusar o
    mesmo `corpo` numa 2ª tentativa duplicava esse prefixo a cada vez
    ("'image-stock'!'image-stock'!O154"), quebrando o range e fazendo toda
    tentativa de retry falhar sempre, mesmo quando o erro original (ex:
    "cota de escrita por minuto") já tinha passado."""
    for tentativa in range(1, tentativas + 1):
        corpo = [
            {"range": gspread.utils.rowcol_to_a1(num_linha, col(nome)), "values": [[valor]]}
            for nome, valor in valores_por_coluna.items()
        ]
        try:
            sheet.batch_update(corpo)
            return True
        except gspread.exceptions.APIError as e:
            print(f"  ⚠️  Erro ao gravar linha {num_linha} (tentativa {tentativa}/{tentativas}): {e}")
            time.sleep(5 * tentativa)
    return False

In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. LOOP PRINCIPAL                                               ║
# ╚══════════════════════════════════════════════════════════════════╝
from tqdm.auto import tqdm

if not (groq_client or mistral_client):
    raise RuntimeError("Nenhuma API disponível — confira GROQ_KEY / MISTRAL_KEY nos Secrets do Colab")

# Duas filas separadas:
# - backfill: já tem Descricao_Cena_PT (rodada com versão antiga do
#   notebook, antes de Tags_Semelhantes existir) mas Tags_Semelhantes_PT
#   ainda vazio -- só falta CALCULAR a partir do que já existe (Tags_PT/EN
#   + Tags_Descricao_PT/EN, se houver), sem gastar IA de novo.
# - completas: nunca foram processadas -- essas sim precisam de IA
#   (download + visão).
fila_backfill = []
fila_completas = []
for num_linha, row in enumerate(dados_tabela, start=2):
    tem_descricao = bool(row.get("Descricao_Cena_PT"))
    tem_semelhantes = bool(row.get("Tags_Semelhantes_PT"))
    if tem_descricao and tem_semelhantes:
        continue  # já 100% processada
    vid_id = str(row.get("ID", "")).strip()
    if not vid_id or vid_id.lower() == "nan":
        continue
    if tem_descricao and not tem_semelhantes:
        fila_backfill.append((num_linha, row))
    else:
        fila_completas.append((num_linha, row))
    if LIMITE_LINHAS is not None and (len(fila_backfill) + len(fila_completas)) >= LIMITE_LINHAS:
        break

print(f"📋 {len(fila_backfill)} linha(s) só de backfill (sem IA) + {len(fila_completas)} linha(s) completas (com IA)")
print()

# ── 5a. Backfill -- sem IA, sem download, só dicionário ──────────────────
processadas_backfill = 0
if fila_backfill:
    print("── Backfill de Tags_Semelhantes (sem gastar IA) ──")
    for num_linha, row in tqdm(fila_backfill, unit="linha"):
        tags_pt      = normalizar_valor_celula(row.get("Tags_PT", ""))
        tags_en      = normalizar_valor_celula(row.get("Tags_EN", ""))
        tags_desc_pt = normalizar_valor_celula(row.get("Tags_Descricao_PT", ""))
        tags_desc_en = normalizar_valor_celula(row.get("Tags_Descricao_EN", ""))

        tags_base_pt = [t.strip() for t in f"{tags_pt},{tags_desc_pt}".split(",") if t.strip()]
        tags_base_en = [t.strip() for t in f"{tags_en},{tags_desc_en}".split(",") if t.strip()]
        tags_semelhantes_pt = normalizar_valor_celula(expandir_tags_semelhantes(tags_base_pt, idioma="pt"))
        tags_semelhantes_en = normalizar_valor_celula(expandir_tags_semelhantes(tags_base_en, idioma="en"))

        if gravar_linha(num_linha, {"Tags_Semelhantes_PT": tags_semelhantes_pt, "Tags_Semelhantes_EN": tags_semelhantes_en}):
            processadas_backfill += 1
    print(f"✅ Backfill: {processadas_backfill}/{len(fila_backfill)}\n")

# ── 5b. Completas -- download + IA (visão) ────────────────────────────────
estado_provedor = {"atual": "mistral"}  # Mistral primeiro -- Groq voltou 0% de resposta valida (content vazio) numa rodada de teste de visao
processadas, falhas, sem_midia = 0, 0, 0

if fila_completas:
    print("── Descrição via IA ──")
    barra = tqdm(fila_completas, unit="linha")
    for num_linha, row in barra:
        vid_id = str(row.get("ID"))
        tags_orig = str(row.get("Tags", ""))

        barra.set_postfix_str(f"ID {vid_id}")

        # busca URL fresca na API (a guardada na planilha já pode ter expirado --
        # URLs da Pixabay só valem 24h) e baixa em seguida
        caminho = baixar_imagem(vid_id)
        if not caminho:
            tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha no download")
            sem_midia += 1
            time.sleep(DELAY_SEGUNDOS)
            continue

        b64_list = [imagem_para_b64(caminho)]
        deletar_temp(caminho)

        prompt = f"ID: {vid_id}\nTags Originais: {tags_orig}\n{PROMPT_SISTEMA}"

        try:
            dados, provedor_usado = preencher_com_ia(prompt, b64_list, estado_provedor)
        except Exception as e:
            tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha total (Groq + Mistral): {e}")
            falhas += 1
            time.sleep(DELAY_SEGUNDOS)
            continue

        tags_pt        = normalizar_valor_celula(dados.get("tags_pt", ""))
        tags_en        = normalizar_valor_celula(dados.get("tags_en", ""))
        desc_pt        = normalizar_valor_celula(dados.get("descricao_cena_pt", ""))
        desc_en        = normalizar_valor_celula(dados.get("descricao_cena_en", ""))
        tags_desc_pt   = normalizar_valor_celula(dados.get("tags_descricao_pt", ""))
        tags_desc_en   = normalizar_valor_celula(dados.get("tags_descricao_en", ""))

        # Tags_Semelhantes = Tags_PT/EN + Tags_Descricao_PT/EN, todas expandidas
        # por sinônimo (dicionario_sinonimos.py, sem IA) -- é isso que o painel
        # de revisão usa pra casar com o lado da Bíblia (versiculo_tags).
        # normalizar_valor_celula ANTES de juntar -- a IA às vezes devolve lista
        # em vez de string (ver normalizar_valor_celula), juntar direto quebraria
        tags_base_pt = [t.strip() for t in f"{tags_pt},{tags_desc_pt}".split(",") if t.strip()]
        tags_base_en = [t.strip() for t in f"{tags_en},{tags_desc_en}".split(",") if t.strip()]
        tags_semelhantes_pt = normalizar_valor_celula(expandir_tags_semelhantes(tags_base_pt, idioma="pt"))
        tags_semelhantes_en = normalizar_valor_celula(expandir_tags_semelhantes(tags_base_en, idioma="en"))

        valores = {
            "Tags_PT": tags_pt, "Tags_EN": tags_en,
            "Descricao_Cena_PT": desc_pt, "Descricao_Cena_EN": desc_en,
            "Tags_Descricao_PT": tags_desc_pt, "Tags_Descricao_EN": tags_desc_en,
            "Tags_Semelhantes_PT": tags_semelhantes_pt, "Tags_Semelhantes_EN": tags_semelhantes_en,
        }

        if gravar_linha(num_linha, valores):
            processadas += 1
        else:
            falhas += 1
            tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha ao gravar na planilha (será retentada na próxima rodada)")

        time.sleep(DELAY_SEGUNDOS)

print()
print("=" * 60)
print("🎉 PROCESSAMENTO CONCLUÍDO")
print("=" * 60)
print(f"   ✅ Backfill (sem IA):     {processadas_backfill}")
print(f"   ✅ Completas (com IA):    {processadas}")
print(f"   ⚠️  Sem mídia:            {sem_midia}")
print(f"   ❌ Falhas:                {falhas}")
print("=" * 60)


📋 0 linha(s) só de backfill (sem IA) + 77 linha(s) completas (com IA)

── Descrição via IA ──


  0%|          | 0/77 [00:00<?, ?linha/s]

  ⚠️  Falha no groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.6-27b` in organization `org_01kmnqhr61ewatdbypq219t6d9` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198447, Requested 3990. Please try again in 17m32.783999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  🚫 groq: cota DIÁRIA esgotada — não tenta mais ele nesta execução.
  ⏳ 429 (Too Many Requests) da Pixabay -- esperando 5s (tentativa 1/4)
  ⏳ 429 (Too Many Requests) da Pixabay -- esperando 5s (tentativa 1/4)

🎉 PROCESSAMENTO CONCLUÍDO
   ✅ Backfill (sem IA):     0
   ✅ Completas (com IA):    77
   ⚠️  Sem mídia:            0
   ❌ Falhas:                0
